In [26]:
from langchain_community.document_loaders import TextLoader, DataFrameLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma


In [27]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# 1. Load your API key
load_dotenv()

# 2. This is the code you asked about (The "Chat/Brain" part)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite", 
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# 3. You ALSO need this for the Vector Database (The "Embedding" part)
# embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-004")
try:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
    # Test it with a tiny query
    embeddings.embed_query("test")
    print("✅ Using text-embedding-004")
except Exception:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
    print("⚠️ Fallback: Using embedding-001")


✅ Using text-embedding-004


In [28]:
import pandas as pd 

books=pd.read_csv("books_cleaned.csv")

books["tagged_description"].to_csv("tagged_descriptions.text",
                                    sep="\n" , index=False, header=False)

In [29]:
# 1. Ensure TextLoader is imported (Fixes the NameError from before)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

raw_data = TextLoader("tagged_descriptions.text").load()

# 2. Set chunk_size to 1000 (roughly the length of a book summary)
# and chunk_overlap to 100 (keeps context between chunks)
text_splitter = CharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100, 
    separator="\n"
)

# 3. This will now run without the ValueError
documents = text_splitter.split_documents(raw_data)



Created a chunk of size 1170, which is longer than the specified 1000
Created a chunk of size 1216, which is longer than the specified 1000
Created a chunk of size 1090, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1269, which is longer than the specified 1000
Created a chunk of size 2012, which is longer than the specified 1000
Created a chunk of size 1227, which is longer than the specified 1000
Created a chunk of size 1186, which is longer than the specified 1000
Created a chunk of size 1216, which is longer than the specified 1000
Created a chunk of size 1193, which is longer than the specified 1000
Created a chunk of size 1059, which is longer than the specified 1000
Created a chunk of size 1272, which is longer than the specified 1000
Created a chunk of size 1637, which is longer than the specified 1000
Created a chunk of size 1134, which is longer than the specified 1000
Created a chunk of s

In [ ]:
# import os
# import shutil
# import time
# import chromadb
# from langchain_chroma import Chroma

# # 1. NEW NAME to avoid the OS lock
# db_path = "./book_db_final_version" 

# # 2. Force delete if it somehow exists
# if os.path.exists(db_path):
#     shutil.rmtree(db_path)

# # 3. Create the first batch
# print("Starting initial ingestion...")
# db_books = Chroma.from_documents(
#     documents=documents[:40], 
#     embedding=embeddings, 
#     persist_directory=db_path
# )

# # 4. Batch loop
# batch_size = 40
# for i in range(batch_size, len(documents), batch_size):
#     batch = documents[i : i + batch_size]
#     db_books.add_documents(batch)
#     print(f"✅ Progress: {i + len(batch)} books stored.")
#     time.sleep(10) # Essential for Gemini Free Tier

Starting initial ingestion...
✅ Progress: 80 books stored.
✅ Progress: 120 books stored.
✅ Progress: 160 books stored.
✅ Progress: 200 books stored.
✅ Progress: 240 books stored.
✅ Progress: 280 books stored.
✅ Progress: 320 books stored.
✅ Progress: 360 books stored.
✅ Progress: 400 books stored.
✅ Progress: 440 books stored.
✅ Progress: 480 books stored.
✅ Progress: 520 books stored.
✅ Progress: 560 books stored.
✅ Progress: 600 books stored.
✅ Progress: 640 books stored.
✅ Progress: 680 books stored.
✅ Progress: 720 books stored.
✅ Progress: 760 books stored.
✅ Progress: 800 books stored.
✅ Progress: 840 books stored.
✅ Progress: 880 books stored.
✅ Progress: 920 books stored.
✅ Progress: 960 books stored.
✅ Progress: 1000 books stored.
✅ Progress: 1040 books stored.
✅ Progress: 1080 books stored.
✅ Progress: 1120 books stored.
✅ Progress: 1160 books stored.
✅ Progress: 1200 books stored.
✅ Progress: 1240 books stored.
✅ Progress: 1280 books stored.
✅ Progress: 1320 books stored.
✅ 

In [44]:
query = "A book about Roman history"
results = db_books.similarity_search(query, k=10)
# print("results:")


In [45]:
# .replace('"', '') removes any double quotes found in that first segment
isbn_str = results[0].page_content.split(" ")[0].replace('"', '').strip()
books[books["isbn13"] == int(isbn_str)]
# print(results[0].metadata)

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
1753,9780380710829,038071082X,The Grass Crown,Colleen McCullough,Fiction,http://books.google.com/books/content?id=Kt3gN...,1992.0,4.29,1104.0,9397.0,The Grass Crown,9780380710829 | The lives of ancient Rome's me...


In [46]:
def retrieve_symmentaic_recommendations(
        query: str,
        top_k: int = 10
    ,
)-> pd.DataFrame:
    recs=db_books.similarity_search(query, k=top_k)
    book_list=[]
    for i in range(len(recs)):
        isbn_str = recs[i].page_content.split(" ")[0].replace('"', '').strip()
        book_info=books[books["isbn13"] == int(isbn_str)]
        book_list.append(book_info)
#    return pd.concat(book_list).reset_index(drop=True)
    return pd.concat(book_list).reset_index(drop=True)
    

In [48]:
results_df=retrieve_symmentaic_recommendations("A book about Fantasy world",top_k=10)
results_df

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780061052392,0061052396,Realms of Dragons,Margaret Weis;Denise Little;Tracy Hickman,Fiction,http://books.google.com/books/content?id=ieAFA...,1999.0,4.04,218.0,47.0,Realms of Dragons: The Worlds of Weis and Hickman,"9780061052392 | In the tradition of ""The Wheel..."
1,9781565047082,1565047087,Changeling Storytellers Guide,Mark Hunter;Nancy Schultz-Yetter;Steve Kenson;...,Games,http://books.google.com/books/content?id=3sMLA...,1998.0,3.37,139.0,27.0,Changeling Storytellers Guide,"9781565047082 | The gates to Arcadia, the orig..."
2,9780345483898,0345483898,Jarka Ruus,Terry Brooks,Fiction,http://books.google.com/books/content?id=5e9ER...,2005.0,3.97,416.0,11867.0,Jarka Ruus,9780345483898 | Twenty years after Grianne Ohm...
3,9780755305308,0755305302,Girl in Hyacinth Blue,Susan Vreeland,American fiction,http://books.google.com/books/content?id=CHg8P...,2002.0,3.73,180.0,136.0,Girl in Hyacinth Blue,9780755305308 | Girl in Hyacinth Blue tells th...
4,9780586066881,0586066888,A Darkness at Sethanon,Raymond E. Feist,English fiction,http://books.google.com/books/content?id=QT078...,1987.0,4.19,527.0,49946.0,A Darkness at Sethanon,9780586066881 | An evil wind blows through Mid...
5,9780061020575,0061020575,Well of Darkness,Margaret Weis;Tracy Hickman,Fiction,http://books.google.com/books/content?id=32gVl...,2001.0,3.66,562.0,2010.0,Well of Darkness: Volume One of the Sovereign ...,9780061020575 | Second in line for succession ...
6,9781857237641,1857237641,Otherland,Tad Williams,Otherland (Imaginary place),http://books.google.com/books/content?id=s73lH...,1998.0,4.01,796.0,192.0,Otherland: River of blue fire,"9781857237641 | Otherland, an incredibly compl..."
7,9780743426923,0743426924,The Diablo: The Kingdom of Shadow,Richard A. Knaak,Fiction,http://books.google.com/books/content?id=aedR1...,2002.0,3.94,339.0,1144.0,The Diablo: The Kingdom of Shadow,"9780743426923 | Since the beginning of time, t..."
8,9780812521351,0812521358,Hart's Hope,Orson Scott Card,Fiction,http://books.google.com/books/content?id=rchTP...,1988.0,3.47,261.0,32.0,Hart's Hope,9780812521351 | Palicroval the Fair spares the...
9,9780756403140,0756403146,Black Sun Rising,C. S. Friedman,Fiction,http://books.google.com/books/content?id=22VxN...,2005.0,3.93,496.0,15265.0,Black Sun Rising,"9780756403140 | On the distant world of Erna, ..."


In [51]:
books['categories'].value_counts().reset_index().query('count>50')

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
5,Religion,117
6,Philosophy,117
7,Comics & Graphic Novels,116
8,Drama,86
9,Juvenile Nonfiction,57


In [52]:
category_mapping = {'Fiction': "Fiction",
'Juvenile Fiction': "Children's Fiction",
'Biography & Autobiography': "Nonfiction",
'History': "Nonfiction",
'Literary Criticism': "Nonfiction",
'Philosophy': "Nonfiction",
'Religion': "Nonfiction",
'Comics & Graphic Novels': "Fiction",
'Drama': "Fiction",
'Juvenile Nonfiction': "Children's Nonfiction",
'Science': "Nonfiction",
'Poetry': "Fiction"}

books['broad_category']=books['categories'].map(category_mapping)


In [53]:
books[~(books['broad_category'].isna())]

,isbn13,isbn10,title,authors,categories,thumbnail,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,broad_category
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 | A NOVEL THAT READERS and criti...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 | A memorable, mesmerizing heroi...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079 | Tricked once more by his wily ...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006 | Joe and his parents are enjoyi...,Children's Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,2002.0,3.71,352.0,8852.0,Taken at the Flood,9780007121014 | A Few Weeks After Marrying An ...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,"9781933648279 | ""Cornell Woolrich's novels def...",Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969 | Rescued from the lockers in wh...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850 | This book is the story of a yo...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 | This collection of the timeles...,Nonfiction


In [68]:
def classify_book_with_gemini(description):
    if not description or len(str(description)) < 10:
        return "Other"

    # Define your focus categories here (equivalent to fiction_categories in the video)
    categories = ["Fiction", "Nonfiction"]

    # The prompt acts as the "Zero-Shot" instruction
    prompt = f"""
    You are an expert book librarian. 
    Classify the following book description into one of these specific categories: {', '.join(categories)}.
    
    Constraint: Your output must be exactly one of those two words.
    
    Description: {description}
    """
    
    try:
        response = llm.invoke(prompt)
        result = response.content.strip()
        
        # Double check that Gemini didn't give extra text
        if "Fiction" in result: return "Fiction"
        if "Nonfiction" in result: return "Nonfiction"
        return "Other"
        
    except Exception as e:
        print(f"Error: {e}")
        return "Other"

In [67]:
sequence=books.loc[books["broad_category"]=="Fiction", "tagged_description"].reset_index(drop=True)[0]

In [69]:
import time

# Identify only the books that currently have 'Other' or are missing a label
to_process = books[books['broad_category'] == "Other"].copy()

print(f"Processing {len(to_process)} books with Gemini...")

for index, row in to_process.iterrows():
    # Call the new function
    new_label = classify_book_with_gemini(row['description'])
    
    # Update the main dataframe
    books.at[index, 'broad_category'] = new_label
    
    print(f"Book: {row['title'][:30]}... -> Classified as: {new_label}")
    
    # IMPORTANT: Wait 4-5 seconds to avoid the 'Rate Limit' error
    time.sleep(5)

# Save the updated data so you don't lose it
books.to_csv("books_fully_classified.csv", index=False)

Processing 0 books with Gemini...


In [79]:
# View a random sample of 10 books and their new labels
test_sample = books[['title', 'broad_category', 'tagged_description']].sample(5)
display(test_sample)

,title,broad_category,tagged_description
4835,The Strangers in the House,Fiction,"9781590171943 | Dirty, drunk, unloved, and unl..."
4743,A Christmas Carol,Fiction,9781580495790 | This Prestwick House Literary ...
1133,23 Days in July,Other,9780306814556 | Taking place over twenty-three...
1774,Only Mine,Fiction,9780380763399 | Whether she′s creating incompa...
2465,Shattered Bonds,Other,9780465070596 | Identifies a disproportionate ...


In [ ]:
# --- STEP 1: PREPARE DOCUMENTS ---
from langchain_core.documents import Document

documents = []
for index, row in books.iterrows():
    doc = Document(
        page_content=row['tagged_description'],
        metadata={
            "title": row['title'],
            "authors": row['authors'],
            "broad_category": row['broad_category']
        }
    )
    documents.append(doc)

print(f"✅ Created {len(documents)} documents ready for the DB.")

✅ Created 5197 documents ready for the DB.


In [93]:
import chromadb
from langchain_chroma import Chroma
import time
# --- PART A: INITIALIZE ---
# We use the raw chromadb client to "touch" the file first
# client = chromadb.PersistentClient(path=db_path)
db_path = "final_books_db" 
print("✅ Native client connected successfully.")

# --- PART B: START THE LANGCHAIN DB ---
# Now we pass that existing client to LangChain
db_books = Chroma.from_documents(
    documents=documents[:40], 
    embedding=embeddings, 
    persist_directory=db_path
)

# --- PART C: BATCH LOOP ---
batch_size = 40
for i in range(batch_size, len(documents), batch_size):
    batch = documents[i : i + batch_size]
    db_books.add_documents(batch)
    print(f"✅ Stored up to {i + len(batch)} books.")
    time.sleep(10)

✅ Native client connected successfully.
✅ Stored up to 80 books.
✅ Stored up to 120 books.
✅ Stored up to 160 books.
✅ Stored up to 200 books.
✅ Stored up to 240 books.
✅ Stored up to 280 books.
✅ Stored up to 320 books.
✅ Stored up to 360 books.
✅ Stored up to 400 books.
✅ Stored up to 440 books.
✅ Stored up to 480 books.
✅ Stored up to 520 books.
✅ Stored up to 560 books.
✅ Stored up to 600 books.
✅ Stored up to 640 books.
✅ Stored up to 680 books.
✅ Stored up to 720 books.
✅ Stored up to 760 books.
✅ Stored up to 800 books.
✅ Stored up to 840 books.
✅ Stored up to 880 books.
✅ Stored up to 920 books.
✅ Stored up to 960 books.
✅ Stored up to 1000 books.
✅ Stored up to 1040 books.
✅ Stored up to 1080 books.
✅ Stored up to 1120 books.
✅ Stored up to 1160 books.
✅ Stored up to 1200 books.
✅ Stored up to 1240 books.
✅ Stored up to 1280 books.
✅ Stored up to 1320 books.
✅ Stored up to 1360 books.
✅ Stored up to 1400 books.
✅ Stored up to 1440 books.
✅ Stored up to 1480 books.
✅ Stored up 

In [95]:
import os
from langchain_chroma import Chroma

# Use the exact same path and embedding function
db_path = "final_books_db"

if os.path.exists(db_path):
    # This only LOADS the database. It does NOT create it or call the API for new embeddings.
    db_books = Chroma(
        persist_directory=db_path,
        embedding_function=embeddings
    )
    print(f"✅ Database loaded successfully with {len(db_books.get()['ids'])} books.")
else:
    print("❌ Error: Database folder not found. Did you delete it?")

✅ Database loaded successfully with 5197 books.
